# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Clustering.** This improves the decision of which treatment — protect, improve CTR, rewrite, merge, prune, or monitor — a batch of pages that share a behavioral profile should get, instead of a content strategist deciding page-by-page from raw metrics.

Across five behavioral features (`ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`), the strongest relationships are moderate at best — `avg_position` and `ctr` at −0.24 (expected: better rank tends to mean more clicks, but position alone explains only a small share of CTR's variation), and `engagement_rate` and `scroll_rate` at 0.26 (expected: engaged visitors tend to scroll more, but the two aren't redundant). Every other pair is near-independent. That means no single threshold or combined score collapses a page's situation — a page can look strong on one axis and weak on another at the same time, which is exactly the case clustering exists for rather than classification (no observed outcome label exists to predict) or ranking (there's no single priority order that fits every treatment).

In [2]:
# This cell is for CODE (numbers, a query, a check).

import pandas as pd
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

visible = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()

candidate_cols = ["ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"]

corr = visible[candidate_cols].corr().round(3)
corr

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct
ctr,1.000,-0.239,0.093,-0.004,-0.001
avg_position,-0.239,1.000,-0.046,0.044,0.025
engagement_rate,0.093,-0.046,1.000,0.258,0.037
scroll_rate,-0.004,0.044,0.258,1.000,0.022
ai_traffic_pct,-0.001,0.025,0.037,0.022,1.000


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**None.** Clustering doesn't predict a label; it groups pages that behave alike based on the features above. What functions as the proxy is downstream: each cluster gets named an archetype (champion, hidden gem, etc.), and that archetype name is what tells the editor which action to take.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Silhouette score (`sklearn.metrics.silhouette_score`) gives a number between −1 and 1 measuring how tight and separated the clusters are. That number alone isn't enough to trust — I'd also pull the mean `ctr`, `avg_position`, and `engagement_rate` per cluster (the same `.groupby().mean()` move used in w01) and check each group reads as a coherent, nameable archetype before assigning any action to it.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row is still one content item. What changes is the population and the column set. The population narrows from 30,000 to 22,006 rows: 1,205 rows have `avg_position == 0`, which the data dictionary defines as "no ranking data" rather than an actual rank, and 7,994 rows have fewer than 100 impressions, where a computed rate (like CTR) is statistically unreliable — a page with 12 impressions and 1 click has a "CTR" of 8.3%, but that's noise, not signal. Including those rows wouldn't broaden the population with real information; it would add unreliable values that could distort cluster centers. The column set narrows to five behavioral rate features (`ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`) plus `content_id`, `client_id`, and `content_type` kept only for identification and grouping, never as clustering inputs. `content_type` is a candidate feature for a future iteration, but as a categorical column it needs encoding before a distance-based method like K-Means can use it.

In [7]:
# This cell is for CODE (numbers, a query, a check).

feature_cols = ["ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"]
id_cols = ["content_id", "client_id", "content_type"]

unit_of_analysis = visible[id_cols + feature_cols].copy()

print(f"Rows before filter: {len(df)}")
print(f"Rows with avg_position == 0 (no ranking data): {(df['avg_position'] == 0).sum()}")
print(f"Rows with impressions_90d < 100: {(df['impressions_90d'] < 100).sum()}")
print(f"Rows kept: {len(unit_of_analysis)}")
unit_of_analysis.head()

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Rows before filter: 30000
Rows with avg_position == 0 (no ranking data): 1205
Rows with impressions_90d < 100: 7994
Rows kept: 22006


,content_id,client_id,content_type,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct
0,content_304f48230142,client_f369cb89fc,keyword article,0.76,10.6,5.88,4.55,0.0
1,content_a1fb4e703a9e,client_4e07408562,keyword article,0.05,20.3,0.00,10.00,0.0
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,0.09,36.5,0.00,28.57,0.0
3,content_331d6c4de07b,client_19581e27de,keyword article,0.49,6.2,1.28,3.45,0.0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,0.13,44.0,0.00,24.29,0.0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A plausible fixed rule — "if CTR is under 1% and average position is worse than 20, flag the page as needing a rewrite" — flags 6,850 pages. But 1,472 of those (21.5%) have above-median engagement rate and above-median scroll rate: people who do find the page read it thoroughly. That's not a content-quality problem, it's a discoverability problem — closer to the "hidden gem" archetype from w01 than "needs rewrite." A two-axis rule can't separate these cases because it only looks at position and CTR; it's blind to the other three behavioral signals that distinguish "bad content" from "good content nobody finds yet."

This is the same finding as the correlation matrix in Section 1, stated as a concrete failure case instead of a coefficient: no single rule collapses five weakly-correlated axes into one correct action, which is what clustering is for.

In [6]:
# This cell is for CODE (numbers, a query, a check).

low_ctr = visible["ctr"] < 1.0
bad_position = visible["avg_position"] > 20
flagged = visible[low_ctr & bad_position]

strong_engagement = flagged[
    (flagged["engagement_rate"] > flagged["engagement_rate"].median()) &
    (flagged["scroll_rate"] > flagged["scroll_rate"].median())
]

print(f"Pages flagged as low-performing by the rule: {len(flagged)}")
print(f"Of those, pages with above-median engagement AND scroll: {len(strong_engagement)} ({len(strong_engagement)/len(flagged):.1%})")

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.